# Repository as Policy with RHO

CaP-X writes a Python robot policy. RHO treats the repository containing that policy as the object to optimize: an agent edits the files, the simulator scores the edited repository, and HELIX keeps only improvements.

In this notebook we will:

1. Start from an **authentic failed Gemma CaP-X cube-stack policy**.
2. Deploy that exact file on five randomly selected, held-out simulator seeds.
3. Run one bounded HELIX/OpenCode evolution generation with local Qwen3 Coder.
4. Deploy the evolved file on the **same five seeds**.
5. Compare task completion, reward, execution failures, and the source diff.

The LLM is used during evolution, not during rollout. Before and after evaluation executes frozen Python policy files.

> This is a small, workshop-sized RHO experiment—not a reproduction of the paper's full multi-generation training run.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(r"""
## Experimental design

A fair before/after comparison needs more than one favorable rollout.

- **Task:** pick up the red cube and stack it on the green cube.
- **Evolution data:** the repository records trial 1 as training and trial 2 as validation.
- **Test data:** five seed/trial IDs are sampled from a separate range and never shown to the evolution agent.
- **Paired comparison:** the same five test seeds are reused after evolution.
- **Frozen deployment:** neither policy file changes during its five rollouts.
- **Failure rule:** a policy that raises an exception receives evaluator reward `0`, even if partial robot motion earned raw simulator reward first.

A fixed meta-seed makes the *selection* of the five test seeds reproducible. MuJoCo, perception, and grasp sampling can still contain nondeterminism, so five trials are a useful smoke test rather than a statistical guarantee.
"""))

In [ ]:
import hashlib
import json
import random
import sys
from pathlib import Path

from IPython.display import Markdown, display

sys.path.insert(0, "/ryzers")
import rho_demo

NOTEBOOK_ROOT = Path("/ryzers/notebooks")
if not NOTEBOOK_ROOT.exists():
    NOTEBOOK_ROOT = Path.cwd()

EXPERIMENT_ROOT = Path("/tmp/rho_five_seed_notebook")
FIXTURE_CODE = NOTEBOOK_ROOT / "fixtures/capx_gemma_e4b_sam3_cube_stack_trial_01.py"
FIXTURE_METADATA = NOTEBOOK_ROOT / "fixtures/capx_gemma_e4b_sam3_cube_stack_trial_01.json"

# Sample once, then reuse this exact list before and after evolution.
EXPERIMENT_META_SEED = 20260822
TEST_SEEDS = random.Random(EXPERIMENT_META_SEED).sample(range(100, 10_000), 5)

GENERATIONS = 1
RHO_TIMEOUT_SECONDS = 480
ROLLOUT_TIMEOUT_SECONDS = 180
CAPTURE_VIDEOS = False  # Set True to retain ten rollout videos.

print("RHO mutation model:", rho_demo.MODEL)
print("Held-out rollout seeds:", TEST_SEEDS)

In [ ]:
display(Markdown(r"""
## Start the evaluator and prepare the repository

The simulator calls four local services:

- **OWLv2** grounds each object phrase into a bounding box.
- **SAM2** segments the selected box into a pixel mask.
- **Contact-GraspNet** proposes a 6-DoF grasp for the red cube.
- **PyRoKi** solves inverse kinematics for the Franka arm.

OWLv2 and SAM2 are ungated and baked into the course image; live evaluation needs neither a Hugging Face token nor a model download. The seed policy remains an unmodified historical CaP-X artifact, including its legacy filename and provenance metadata.

`prepare_workshop` creates a disposable Git repository around the recorded CaP-X output. The generated program is copied verbatim into `solver/program.py`; evaluator code, provenance, API documentation, and HELIX/OpenCode configuration are protected from mutation.
"""))

In [ ]:
servers = rho_demo.ensure_services()

ROOT = rho_demo.prepare_workshop(
    EXPERIMENT_ROOT / "candidate",
    artifact=FIXTURE_CODE,
    provenance=FIXTURE_METADATA,
    heldout_trial=2,
    generations=GENERATIONS,
)

PROVENANCE = json.loads((ROOT / "provenance.json").read_text())
SEED_PROGRAM = (ROOT / "solver/program.py").read_text()
SEED_SHA256 = hashlib.sha256(SEED_PROGRAM.encode()).hexdigest()

print("Candidate repository:", ROOT)
print("Training trial:", PROVENANCE["training_trial"])
print("Validation trial:", PROVENANCE["heldout_trial"])
print("Seed policy SHA-256:", SEED_SHA256)

In [ ]:
print(SEED_PROGRAM)

buggy_lines = [
    line for line in SEED_PROGRAM.splitlines()
    if "green_pose[0][" in line
]
print("\nSuspicious lines:")
for line in buggy_lines:
    print("  ", line)

print("\nAPI contract excerpt:")
print("get_object_pose(...)[0] is already a flat position array with shape (3,).")

In [ ]:
display(Markdown(r"""
## Roll out the policy before evolution

Each call below creates a fresh cube-stack environment, resets it with one test seed, and executes the same `solver/program.py` in a killable child process. No model is queried.

The evaluator reports two rewards:

- `raw_reward`: progress observed by the simulator, even if code later crashes.
- `reward`: the deployability score used by RHO; execution errors are forced to zero.

This distinction prevents a policy from receiving credit for moving the robot partway and then raising an exception.
"""))

In [ ]:
def compact_result(result):
    return {
        key: result.get(key)
        for key in (
            "trial",
            "reward",
            "raw_reward",
            "task_completed",
            "timed_out",
            "stderr",
            "traceback",
            "feedback",
            "video",
        )
    }


def rollout(policy_root, seeds, label):
    results = []
    print(f"{label}: {len(seeds)} frozen-policy rollouts")
    for index, seed in enumerate(seeds, start=1):
        result = rho_demo.score_candidate(
            policy_root,
            "val",
            trial=seed,
            capture=CAPTURE_VIDEOS,
            timeout_seconds=ROLLOUT_TIMEOUT_SECONDS,
        )
        result = compact_result(result)
        results.append(result)
        status = "PASS" if result["task_completed"] else "FAIL"
        print(
            f"  {index}/5 seed={seed}: {status}  "
            f"reward={float(result['reward'] or 0):.4f}  "
            f"raw={float(result['raw_reward'] or 0):.4f}"
        )
    return results


def aggregate(results):
    count = len(results)
    completed = sum(bool(item["task_completed"]) for item in results)
    execution_failures = sum(
        bool(item.get("stderr"))
        or bool(item.get("traceback"))
        or bool(item.get("timed_out"))
        for item in results
    )
    return {
        "trials": count,
        "completed": completed,
        "success_rate": completed / count,
        "mean_reward": sum(float(item["reward"] or 0) for item in results) / count,
        "mean_raw_reward": sum(float(item["raw_reward"] or 0) for item in results) / count,
        "execution_failures": execution_failures,
    }

In [ ]:
before_results = rollout(ROOT, TEST_SEEDS, "BEFORE EVOLUTION")
before_summary = aggregate(before_results)
before_summary

### What failed?

The policy successfully perceives and grasps the red cube, so the simulator can report substantial partial reward. It then evaluates `green_pose[0][2]`.

But `green_pose` has shape `(3,)`. `green_pose[0]` is one scalar X coordinate, and indexing that scalar with `[2]` raises `IndexError`. Because the same shape error is independent of object placement, we expect it to reproduce across seeds.

In [ ]:
print(before_results[0]["feedback"])

assert (ROOT / "solver/program.py").read_text() == SEED_PROGRAM
assert hashlib.sha256((ROOT / "solver/program.py").read_bytes()).hexdigest() == SEED_SHA256
print("\nPolicy integrity check passed: all five baseline runs used identical bytes.")

## Evolve the repository

`run_helix` now starts one bounded evolution generation:

1. **Seed evaluation:** HELIX scores the unmodified Git repository on its configured training example.
2. **Mutation proposal:** OpenCode gives local Qwen3 Coder read/edit/bash tools inside a worktree. It may edit only files below `solver/`.
3. **Candidate evaluation:** the protected evaluator executes the proposed repository in CaP-X.
4. **Strict-improvement gate:** a child that does not beat its parent is rejected.
5. **Frontier update:** an accepted best child is copied to `live_best/`.

The agent can read the API contract and evaluator traceback, but it does **not** receive `TEST_SEEDS` or the five rollout results above. Evolution uses the repository's training/validation configuration, keeping our five seeds as a held-out test set.

In [ ]:
live_run = rho_demo.run_helix(
    ROOT,
    generations=GENERATIONS,
    timeout_seconds=RHO_TIMEOUT_SECONDS,
)
summary = rho_demo.summarize_run(ROOT)

print(
    f"HELIX exit={live_run.returncode}  timed_out={live_run.timed_out}  "
    f"elapsed={live_run.elapsed_seconds:.1f}s"
)
print("Accepted mutation:", summary["accepted"])
print("Improved best candidate:", summary["improved_best"])

if not summary["improved_best"]:
    raise RuntimeError(
        "No improved live candidate was produced. Inspect the HELIX output above "
        "and rerun this cell before continuing."
    )

EVOLVED_ROOT = Path(summary["live_best"])
EVOLVED_PROGRAM = (EVOLVED_ROOT / "solver/program.py").read_text()
EVOLVED_SHA256 = hashlib.sha256(EVOLVED_PROGRAM.encode()).hexdigest()

print("\nSemantic mutation:")
for item in summary["semantic_mutation"]:
    print("  -", item)
print("\nAccepted source diff:\n")
print(summary["best_diff"])
print("Evolved policy SHA-256:", EVOLVED_SHA256)

## Roll out the evolved policy on the same five seeds

Nothing about the test set changes here: same task, same seed order, same evaluator, and same timeout. The only experimental variable is the repository selected by HELIX.

This paired design lets us separate a real code improvement from an easier second batch of scenes.

In [ ]:
after_results = rollout(EVOLVED_ROOT, TEST_SEEDS, "AFTER EVOLUTION")
after_summary = aggregate(after_results)

assert (EVOLVED_ROOT / "solver/program.py").read_text() == EVOLVED_PROGRAM
assert TEST_SEEDS == [item["trial"] for item in before_results]
assert TEST_SEEDS == [item["trial"] for item in after_results]

print("\nBefore:", before_summary)
print("After: ", after_summary)
print("\nPaired seed and evolved-policy integrity checks passed.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(TEST_SEEDS))
width = 0.36
before_rewards = [float(item["reward"] or 0) for item in before_results]
after_rewards = [float(item["reward"] or 0) for item in after_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(x - width / 2, before_rewards, width, label="Before")
axes[0].bar(x + width / 2, after_rewards, width, label="After")
axes[0].set_title("Paired evaluator reward by held-out seed")
axes[0].set_xlabel("Simulator seed / trial ID")
axes[0].set_ylabel("Evaluator reward (0–1)")
axes[0].set_xticks(x, [str(seed) for seed in TEST_SEEDS], rotation=30)
axes[0].set_ylim(0, 1.05)
axes[0].legend()

summary_labels = ["Completion rate", "Execution failures / 5"]
axes[1].bar(
    np.arange(2) - width / 2,
    [before_summary["success_rate"], before_summary["execution_failures"] / 5],
    width,
    label="Before",
)
axes[1].bar(
    np.arange(2) + width / 2,
    [after_summary["success_rate"], after_summary["execution_failures"] / 5],
    width,
    label="After",
)
axes[1].set_title("Completion and deployability")
axes[1].set_ylabel("Fraction of five rollouts")
axes[1].set_xticks(np.arange(2), summary_labels)
axes[1].set_ylim(0, 1.05)
axes[1].legend()
fig.tight_layout()
plt.show()

REPORT_PATH = EXPERIMENT_ROOT / "five_seed_report.json"
report = {
    "schema_version": "rho-notebook-five-seed/v1",
    "task": "cube_stack",
    "experiment_meta_seed": EXPERIMENT_META_SEED,
    "test_seeds": TEST_SEEDS,
    "test_seeds_used_by_evolution": False,
    "seed_policy_sha256": SEED_SHA256,
    "evolved_policy_sha256": EVOLVED_SHA256,
    "before": {"aggregate": before_summary, "results": before_results},
    "evolution": {
        "model": rho_demo.MODEL,
        "generations": GENERATIONS,
        "elapsed_seconds": live_run.elapsed_seconds,
        "accepted": summary["accepted"],
        "improved_best": summary["improved_best"],
        "diff": summary["best_diff"],
    },
    "after": {"aggregate": after_summary, "results": after_results},
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2) + "\n")
print("Saved report:", REPORT_PATH)

In [ ]:
before_completed = before_summary["completed"]
after_completed = after_summary["completed"]
crashes_removed = (
    before_summary["execution_failures"] - after_summary["execution_failures"]
)

display(Markdown(f"""
## Interpretation

- **Task completion:** `{before_completed}/5` before → `{after_completed}/5` after.
- **Execution failures:** `{before_summary['execution_failures']}/5` before → `{after_summary['execution_failures']}/5` after.
- **Mean evaluator reward:** `{before_summary['mean_reward']:.3f}` before → `{after_summary['mean_reward']:.3f}` after.
- **Runtime failures removed:** `{crashes_removed}` across the paired rollouts.

The accepted edit generalizes if it removes the shape/indexing crash on seeds that evolution never saw. Task completion can still be below 5/5 because grasp selection, perception, and placement are separate failure modes. RHO repaired the observed program defect; it did not prove that every part of the manipulation strategy is universally robust.

With only five rollouts, report the observed rate rather than treating it as a precise success probability.
"""))

## What this notebook demonstrates

RHO did not update model weights and it did not generate a fresh policy for each test scene. It evolved a versioned software artifact once, then deployed the resulting Python file repeatedly.

That separation is the core repository-as-policy idea:

**generate code → evaluate repository → mutate repository → gate on execution → deploy frozen best repository**

The disposable repository, HELIX worktrees, full evaluator feedback, accepted diff, and five-seed report remain under `/tmp/rho_five_seed_notebook/` for inspection. Call `rho_demo.stop_owned_services()` when you are finished and want to release the perception-service processes.

## End-to-end validation result

This notebook was executed on the workshop GPU on 2026-08-22 with sampled seeds `8740, 1854, 2195, 9455, 2461`.

- Before evolution: **0/5 completed**, **5/5 execution failures**, mean evaluator reward **0.000**.
- HELIX/OpenCode: one mutation accepted in **95.4 seconds**.
- Accepted edit: direct flat-vector indexing (`green_pose[2]`, `green_pose[0]`, `green_pose[1]`).
- After evolution: **4/5 completed**, **0/5 execution failures**, mean evaluator reward **0.804**.

The remaining failed rollout executed cleanly but missed the manipulation objective, which demonstrates the difference between repairing a program defect and solving every physical-control failure mode. A fully executed copy with cell outputs is saved at `experiment_results/05_repository_as_policy_with_rho.executed.ipynb`.